# Multi-Gene Mammalian Species Classification
A modular ML pipeline for classifying mammalian species using multi-gene DNA sequences.

**Pipeline stages:**
1. Imports & Configuration
2. Data Collection (NCBI API)
3. Data Cleaning & Filtering
4. Per-Gene FASTA Export
5. Sequence Quality Filtering
6. Species Intersection
7. Sequence Alignment (MAFFT + TrimAl)
8. Feature Engineering (K-mer, One-Hot, BioVec)
9. Phylogenetic Tree & Distance Matrix
10. Metadata Preparation
11. Hierarchical Clustering
12. Multi-Input CNN Classification

## 0. Installation

In [ ]:
!pip install biopython ete3 gensim geopy torch torchvision

## 1. Imports & Configuration

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import sleep
from collections import defaultdict, Counter
from itertools import product
import subprocess
import requests

# BioPython
from Bio import Entrez, SeqIO, AlignIO, Phylo
from Bio.Align import MultipleSeqAlignment
from Bio.SeqRecord import SeqRecord

# Gensim (BioVec)
from gensim.models import Word2Vec

# ETE toolkit
from ete3 import Tree

# scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, accuracy_score,
    precision_recall_fscore_support,
    adjusted_rand_score, normalized_mutual_info_score,
)

# scipy
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Geocoding
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
# ── Directory paths (edit BASE_DIR to match your project root) ────────────────
BASE_DIR       = "."
RAW_FASTA_DIR  = os.path.join(BASE_DIR, "gene_fastas")
FILTERED_DIR   = os.path.join(BASE_DIR, "filtered")
EXTRACTED_DIR  = os.path.join(BASE_DIR, "extracted_sequences")
ALIGNED_DIR    = os.path.join(BASE_DIR, "aligned")
KMER_DIR       = os.path.join(BASE_DIR, "kmers")
ONEHOT_DIR     = os.path.join(BASE_DIR, "onehot")
BIOVEC_DIR     = os.path.join(BASE_DIR, "biovec")

# ── External tool paths (update if not on PATH) ───────────────────────────────
MAFFT_PATH   = "mafft"    # e.g. r"C:\mafft-win\mafft.bat"
TRIMAL_PATH  = "trimal"   # e.g. r"C:\tools\trimal\trimal.exe"

# ── Gene / species lists ──────────────────────────────────────────────────────
SELECTED_GENES = ["COI", "CytB", "16S rRNA", "RAG1", "BRCA1", "APOB"]
GENES          = ["apob", "brca1", "coi", "cytb", "rag1"]   # post-filtering (lowercase)

GENE_SYNONYMS = {
    "COI":      ["COI", "cytochrome c oxidase subunit I", "cox1"],
    "CytB":     ["CytB", "cytochrome b"],
    "16S rRNA": ["16S rRNA", "16S ribosomal RNA"],
    "RAG1":     ["RAG1"],
    "BRCA1":    ["BRCA1"],
    "APOB":     ["APOB"],
}

MAMMALS = [
    # Primates
    "Homo sapiens", "Pan troglodytes", "Gorilla gorilla", "Pongo pygmaeus",
    "Macaca mulatta", "Ateles geoffroyi", "Cercopithecus aethiops", "Callithrix jacchus",
    "Saimiri sciureus", "Lemur catta", "Tarsius syrichta", "Aotus trivirgatus",
    "Brachyteles arachnoides", "Papio hamadryas", "Cebus capucinus", "Lagothrix lagothricha",
    "Alouatta palliata", "Nasalis larvatus", "Colobus guereza", "Pithecia pithecia",
    "Saguinus oedipus", "Trachypithecus francoisi", "Gibbon", "Bornean orangutan", "Gorilla beringei",
    # Rodents
    "Mus musculus", "Rattus norvegicus", "Spermophilus tridecemlineatus", "Sciurus carolinensis",
    "Castor canadensis", "Cavia porcellus", "Hystrix cristata", "Marmota monax", "Dipodomys ordii",
    "Peromyscus maniculatus", "Octodon degus", "Cricetulus griseus", "Meriones unguiculatus",
    "Erethizon dorsatum", "Neotoma albigula", "Rattus rattus", "Microtus arvalis",
    "Arvicola amphibius", "Chinchilla lanigera", "Myocastor coypus", "Ctenomys",
    "Acomys cahirinus", "Phodopus sungorus",
    # Carnivores
    "Panthera leo", "Panthera tigris", "Canis lupus", "Ursus maritimus", "Vulpes vulpes",
    "Procyon lotor", "Mustela putorius furo", "Lynx lynx", "Gulo gulo", "Mephitis mephitis",
    "Ailurus fulgens", "Felis catus", "Canis lupus familiaris", "Martes martes",
    "Herpestes javanicus", "Nyctereutes procyonoides", "Bassariscus astutus",
    "Otaria flavescens", "Enhydra lutris", "Mustela erminea", "Lutra lutra", "Hyaena hyaena",
    # Ungulates
    "Cervus elaphus", "Equus caballus", "Bos taurus", "Capra hircus", "Ovis aries",
    "Bubalus bubalis", "Alces alces", "Antilocapra americana", "Giraffa camelopardalis",
    "Rangifer tarandus", "Hippopotamus amphibius", "Camelus dromedarius", "Equus asinus",
    "Gazella gazella", "Bison bison", "Equus przewalskii", "Dama dama",
    # Bats
    "Pteropus vampyrus", "Tadarida brasiliensis", "Myotis lucifugus", "Pipistrellus pipistrellus",
    "Eptesicus fuscus", "Vespertilio murinus", "Miniopterus schreibersii",
    "Rhinolophus ferrumequinum",
]

# ── Model hyperparameters ─────────────────────────────────────────────────────
NUM_CLASSES   = 5
BATCH_SIZE    = 8
EPOCHS        = 15
LEARNING_RATE = 1e-3
KMER_K        = 3
BIOVEC_DIM    = 100

# ── IUCN ordinal mapping ──────────────────────────────────────────────────────
IUCN_ORDER = {
    "LC":  0, "NT": 1, "VU": 2, "EN": 3,
    "CR":  4, "EW": 5, "EX": 6, "NE": -1,
}

# ── Accession → species name mapping (used in k-mer feature step) ─────────────
ACCESSION_TO_SPECIES = {
    "XM_004028908.4": "Gorilla gorilla gorilla",
    "XM_001097500.4": "Macaca mulatta",
    "XM_035273096.2": "Callithrix jacchus",
    "XM_045549123.1": "Lemur catta",
    "XM_033176393.1": "Trachypithecus francoisi",
    "NM_009693.2":    "Mus musculus",
    "XM_005322645.4": "Ictidomys tridecemlineatus",
    "JN414051.1":     "Castor canadensis",
    "XM_013013544.1": "Dipodomys ordii",
    "XM_032909188.1": "Rattus rattus",
    "JN414052.1":     "Chinchilla lanigera",
    "JN414050.1":     "Myocastor coypus",
    "XM_042933585.1": "Panthera leo",
    "XM_042982424.1": "Panthera tigris",
    "XM_040623722.1": "Ursus maritimus",
    "XM_055327418.1": "Nyctereutes procyonoides",
    "XM_039222451.1": "Hyaena hyaena",
    "XM_018055590.1": "Capra hircus",
    "XM_027966544.2": "Ovis aries",
    "XM_057743096.1": "Hippopotamus amphibius kiboko",
    "XM_044772007.2": "Equus asinus",
    "XM_070573197.1": "Equus przewalskii",
    "XM_054728236.1": "Eptesicus fuscus",
    "XM_033125573.1": "Rhinolophus ferrumequinum",
}

## 2. Data Collection (NCBI API)

In [ ]:
# Replace with your credentials
Entrez.email   = "your_email@example.com"
Entrez.api_key = "your_api_key"

os.makedirs(RAW_FASTA_DIR, exist_ok=True)
dna_data = {}

for species in MAMMALS:
    species_data = {"dna_sequences": []}
    print(f"\nFetching sequences for {species}...")

    for gene in SELECTED_GENES:
        gene_entry = {"gene": gene, "sequence": None}
        found = False

        for gene_term in GENE_SYNONYMS[gene]:
            if gene in ["COI", "CytB", "16S rRNA"]:
                query = f"({species}[Organism]) AND ({gene_term}[Gene]) AND mitochondrion"
            else:
                query = f"({species}[Organism]) AND ({gene_term}[Gene])"

            try:
                handle = Entrez.esearch(db="nucleotide", term=query, retmax=5)
                record = Entrez.read(handle)
                handle.close()

                if record["IdList"]:
                    seq_id = record["IdList"][0]
                    seq_handle = Entrez.efetch(
                        db="nucleotide", id=seq_id, rettype="fasta", retmode="text"
                    )
                    sequence = seq_handle.read()
                    seq_handle.close()

                    gene_entry["sequence"] = sequence
                    fasta_path = os.path.join(RAW_FASTA_DIR, f"{gene}.fasta")
                    with open(fasta_path, "a") as fh:
                        fh.write(sequence + "\n")

                    print(f"  Saved: {species.replace(' ', '_')}_{gene}")
                    found = True
                    break
                else:
                    sleep(0.4)

            except Exception as e:
                print(f"  Error fetching {gene} for {species}: {e}")
                continue

        if not found:
            print(f"  No {gene} sequence found for {species}.")

        species_data["dna_sequences"].append(gene_entry)

    dna_data[species] = species_data

with open("dna_sequences.json", "w") as f:
    json.dump(dna_data, f, indent=2)
print("\nDone. Data saved to dna_sequences.json.")

## 3. Data Cleaning & Filtering

In [ ]:
def check_missing_sequences(dna_data):
    """Return per-species and per-gene missing sequence counts."""
    per_species = {}
    per_gene = defaultdict(int)
    for species, info in dna_data.items():
        missing = sum(1 for e in info["dna_sequences"] if e["sequence"] is None)
        per_species[species] = missing
        for e in info["dna_sequences"]:
            if e["sequence"] is None:
                per_gene[e["gene"]] += 1
    return per_species, per_gene


def print_missing_report(per_species, per_gene):
    print("=== Missing Gene Sequences per Species ===")
    for sp, count in per_species.items():
        print(f"  {sp}: {count} missing")
    print("\n=== Missing Sequences per Gene ===")
    for gene, count in per_gene.items():
        print(f"  {gene}: {count} missing")


# ── Load raw data ─────────────────────────────────────────────────────────────
with open("dna_sequences.json") as f:
    dna_data = json.load(f)

print("--- Initial missing counts ---")
print_missing_report(*check_missing_sequences(dna_data))

# ── Step 1: Remove 16S rRNA (most sequences missing) ─────────────────────────
for species, info in dna_data.items():
    info["dna_sequences"] = [
        e for e in info["dna_sequences"] if e["gene"] != "16S rRNA"
    ]

print("\n--- After removing 16S rRNA ---")
print_missing_report(*check_missing_sequences(dna_data))

# ── Step 2: Drop species missing 2+ sequences ────────────────────────────────
dna_data = {
    sp: info for sp, info in dna_data.items()
    if sum(1 for e in info["dna_sequences"] if e["sequence"] is None) < 2
}
print(f"\nSpecies remaining after <2-missing filter: {len(dna_data)}")

# ── Step 3: Keep only species with ALL sequences present ─────────────────────
complete_data = {
    sp: info for sp, info in dna_data.items()
    if all(e["sequence"] is not None for e in info["dna_sequences"])
}
print(f"Species with complete gene data: {len(complete_data)}")

with open("complete_dna_sequences.json", "w") as f:
    json.dump(complete_data, f, indent=2)
print("Saved → complete_dna_sequences.json")

## 4. Per-Gene FASTA Export

In [ ]:
with open("complete_dna_sequences.json") as f:
    dna_data = json.load(f)

os.makedirs(RAW_FASTA_DIR, exist_ok=True)
gene_sequences = defaultdict(list)

for species, info in dna_data.items():
    for entry in info["dna_sequences"]:
        gene = entry["gene"]
        header = f">{species.replace(' ', '_')}"
        cleaned_seq = "".join(entry["sequence"].splitlines()[1:])
        gene_sequences[gene].append(f"{header}\n{cleaned_seq}")

for gene, sequences in gene_sequences.items():
    fasta_path = os.path.join(RAW_FASTA_DIR, f"{gene}.fasta")
    with open(fasta_path, "w") as f:
        f.write("\n".join(sequences) + "\n")

print(f"Wrote {len(gene_sequences)} gene FASTA files to '{RAW_FASTA_DIR}/'")

## 5. Sequence Quality Filtering

In [ ]:
def print_sequence_lengths(sequences, gene_name):
    lengths = [len(seq.seq) for seq in sequences]
    print(f"  {gene_name}: {len(lengths)} seqs | "
          f"min={min(lengths)}, max={max(lengths)}, "
          f"median={np.median(lengths):.0f}, mean={np.mean(lengths):.0f}")


def filter_sequences(input_fasta, output_fasta, max_ambiguous=20):
    """Filter sequences by length and ambiguity; save passing sequences."""
    sequences = list(SeqIO.parse(input_fasta, "fasta"))
    if not sequences:
        print(f"[!] No sequences in {input_fasta}")
        return

    gene_name = os.path.basename(input_fasta).split(".")[0].lower()
    gene_params = {
        "coi":   {"length_cap": 2000,  "min_length_ratio": 0.4},
        "cytb":  {"length_cap": 2000,  "min_length_ratio": 0.4},
        "rag1":  {"length_cap": 10000, "min_length_ratio": 0.1},
        "brca1": {"length_cap": 10000, "min_length_ratio": 0.1},
        "apob":  {"length_cap": 12000, "min_length_ratio": 0.1},
    }
    params = gene_params.get(gene_name, {"length_cap": 5000, "min_length_ratio": 0.4})

    clipped = [min(len(s.seq), params["length_cap"]) for s in sequences]
    length_threshold = int(params["min_length_ratio"] * np.percentile(clipped, 95))

    print_sequence_lengths(sequences, gene_name)

    filtered, too_short, too_ambiguous = [], 0, 0
    for record in sequences:
        seq = str(record.seq).upper()
        if len(seq) < length_threshold:
            too_short += 1
            continue
        if seq.count("N") > max_ambiguous:
            too_ambiguous += 1
            continue
        filtered.append(record)

    os.makedirs(os.path.dirname(output_fasta), exist_ok=True)
    if filtered:
        SeqIO.write(filtered, output_fasta, "fasta")
        print(f"  [✓] {gene_name}: {len(filtered)} sequences saved → {output_fasta}")
    else:
        print(f"  [!] No sequences passed for {gene_name}")
    print(f"      Skipped: {too_short} (length), {too_ambiguous} (ambiguity)")

In [ ]:
os.makedirs(FILTERED_DIR, exist_ok=True)

for gene in ["COI", "CytB", "RAG1", "BRCA1", "APOB"]:
    input_path  = os.path.join(RAW_FASTA_DIR, f"{gene}.fasta")
    output_path = os.path.join(FILTERED_DIR,  f"{gene}.filtered.fasta")
    print(f"\n[→] {gene}")
    filter_sequences(input_path, output_path)

## 6. Species Intersection
Find the set of species with sequences in all five genes.

In [ ]:
# Raw FASTA description sets per gene (from filtered files)
rag1_species = set([
    'Acomys cahirinus isolate', 'Ailurus fulgens', 'Alouatta palliata coibensis isolate',
    'Aotus trivirgatus isolate', 'Arvicola amphibius recombination activating',
    'Ateles geoffroyi vellerosus isolate', 'Bassariscus astutus recombination',
    'Bos taurus recombination activating', 'Brachyteles arachnoides isolate',
    'Callithrix jacchus recombination activating', 'Canis lupus baileyi recombination activating',
    'Capra hircus recombination activating', 'Carcharodon carcharias recombination activating',
    'Castor canadensis recombination activating protein', 'Cavia porcellus recombination activating',
    'Chinchilla lanigera voucher', 'Chlorocebus aethiops isolate', 'Colobus guereza isolate',
    'Dipodomys ordii recombination activating gene', 'Enhydra lutris',
    'Eptesicus fuscus recombination activating', 'Equus asinus recombination activating',
    'Equus przewalskii recombination activating', 'Erethizon dorsatum recombination activating protein',
    'Gorilla gorilla gorilla recombination activating', 'Herpestes javanicus recombination activating gene',
    'Hippopotamus amphibius kiboko recombination activating', 'Homo sapiens recombination activating',
    'Hyaena hyaena recombination activating', 'Ictidomys tridecemlineatus recombination activating',
    'Lagothrix lagotricha isolate', 'Lemur catta recombination activating',
    'Macaca mulatta recombination activating', 'Marmota monax isolate', 'Martes martes',
    'Mephitis mephitis', 'Microtus arvalis voucher', 'Mus musculus recombination activating',
    'Mustela erminea recombination activating', 'Myocastor coypus recombination activating protein',
    'Nasalis larvatus isolate', 'Nyctereutes procyonoides recombination activating',
    'Ovis aries recombination activating', 'Pan troglodytes recombination activating',
    'Panthera leo recombination activating', 'Panthera tigris recombination activating',
    'Pithecia pithecia isolate', 'Pongo pygmaeus recombination activating', 'Procyon lotor',
    'Pteropus giganteus recombination activating', 'Rattus norvegicus recombination activating',
    'Rattus rattus recombination activating protein', 'Rhinolophus ferrumequinum recombination activating',
    'Trachypithecus francoisi recombination activating', 'Ursus maritimus recombination activating',
    'Vulpes vulpes recombination activating',
])
brca1_species = set([
    'Ailurus fulgens brca', 'Antilocapra americana voucher', 'Arvicola amphibius', 'Bubalus bubalis',
    'Callithrix jacchus', 'Canis lupus familiaris', 'Capra hircus', 'Castor canadensis', 'Cavia porcellus',
    'Chinchilla lanigera breast cancer', 'Colobus guereza breast cancer type',
    'Ctenomys boliviensis breast and ovarian cancer susceptibility', 'Dama dama', 'Desmodus rotundus',
    'Dipodomys ordii breast cancer', 'Enhydra lutris brca', 'Eptesicus fuscus', 'Equus asinus',
    'Equus caballus', 'Equus przewalskii', 'Erethizon dorsatum breast and ovarian cancer susceptibility',
    'Felis catus', 'Giraffa camelopardalis breast and ovarian cancer susceptibility',
    'Gorilla gorilla gorilla', 'Hippopotamus amphibius kiboko', 'Hyaena hyaena',
    'Ictidomys tridecemlineatus', 'Lemur catta', 'Lutra lutra', 'Macaca mulatta', 'Marmota monax',
    'Mephitis mephitis brca', 'Mus musculus breast cancer', 'Myocastor coypus common coypu breast and ovarian cancer susceptibility',
    'Myotis lucifugus', 'Nyctereutes procyonoides', 'Octodon degus', 'Ovis aries', 'Pan troglodytes',
    'Panthera leo', 'Panthera tigris', 'Peromyscus maniculatus bairdii', 'Phodopus sungorus breast cancer protein',
    'Pongo pygmaeus', 'Pteropus giganteus', 'Rattus norvegicus', 'Rattus rattus breast cancer type',
    'Rhinolophus ferrumequinum', 'Saimiri sciureus breast cancer type', 'Trachypithecus francoisi',
    'Ursus maritimus', 'Vulpes vulpes',
])
apob_species = set([
    'Antilocapra americana apolipoprotein', 'Arvicola amphibius apolipoprotein', 'Bos taurus apolipoprotein',
    'Bubalus bubalis apolipoprotein', 'Callithrix jacchus apolipoprotein', 'Canis lupus baileyi apolipoprotein',
    'Capra hircus apolipoprotein', 'Carlito syrichta apolipoprotein', 'Castor canadensis apolipoprotein',
    'Cavia porcellus apolipoprotein', 'Chinchilla lanigera apolipoprotein',
    'Ctenomys boliviensis apolipoprotein', 'Dama dama isolate', 'Desmodus rotundus apolipoprotein',
    'Dipodomys ordii apolipoprotein', 'Eptesicus fuscus apolipoprotein', 'Equus asinus apolipoprotein',
    'Equus caballus apolipoprotein', 'Equus przewalskii apolipoprotein',
    'Erethizon dorsatum apolipoprotein', 'Felis catus apolipoprotein',
    'Gorilla gorilla gorilla apolipoprotein', 'Hippopotamus amphibius kiboko apolipoprotein',
    'Hyaena hyaena apolipoprotein', 'Ictidomys tridecemlineatus apolipoprotein',
    'Lemur catta apolipoprotein', 'Macaca mulatta apolipoprotein', 'Marmota monax apolipoprotein',
    'Mus musculus apolipoprotein', 'Myocastor coypus apolipoprotein', 'Myotis lucifugus apolipoprotein',
    'Nyctereutes procyonoides apolipoprotein', 'Octodon degus apolipoprotein', 'Ovis aries apolipoprotein',
    'Panthera leo apolipoprotein', 'Panthera tigris apolipoprotein',
    'Peromyscus maniculatus bairdii apolipoprotein', 'Pteropus giganteus apolipoprotein',
    'Rattus rattus apolipoprotein', 'Rhinolophus ferrumequinum apolipoprotein',
    'Saimiri sciureus apolipoprotein', 'Trachypithecus francoisi apolipoprotein',
    'Ursus maritimus apolipoprotein', 'Vulpes vulpes apolipoprotein',
])
coi_species = set([
    'Acomys cahirinus isolate', 'Ailurus fulgens mitochondrion', 'Alouatta palliata isolate',
    'Antilocapra americana isolate', 'Aotus trivirgatus mitochondrion', 'Ateles geoffroyi isolate',
    'Bassariscus astutus voucher', 'Bison bison voucher', 'Bos taurus isolate',
    'Brachyteles arachnoides mitochondrion sequence', 'Bubalus bubalis isolate',
    'Callithrix jacchus mitochondrion', 'Camelus dromedarius isolate', 'Canis lupus isolate',
    'Capra hircus mitochondrion', 'Carcharodon carcharias voucher', 'Carlito syrichta mitochondrion',
    'Castor canadensis mitochondrion', 'Cavia porcellus mitochondrion', 'Chinchilla lanigera mitochondrion',
    'Chlorocebus aethiops mitochondrion', 'Colobus guereza mitochondrion',
    'Cricetulus griseus mitochondrion', 'Dama dama isolate', 'Dipodomys ordii mitochondrion sequence',
    'Enhydra lutris mitochondrion', 'Eptesicus fuscus mitochondrion', 'Equus asinus isolate',
    'Equus caballus mitochondrion', 'Equus przewalskii mitochondrion', 'Felis catus mitochondrion',
    'Gazella gazella isolate', 'Giraffa camelopardalis camelopardalis isolate', 'Gorilla beringei isolate',
    'Gorilla gorilla gorilla mitochondrion', 'Gulo gulo mitochondrion', 'Herpestes javanicus mitochondrion',
    'Hippopotamus amphibius mitochondrion', 'Homo sapiens voucher', 'Hyaena hyaena isolate',
    'Ictidomys tridecemlineatus mitochondrion', 'Lagothrix lagotricha mitochondrion',
    'Lemur catta mitochondrion complete mitochondrial genome', 'Lutra lutra isolate',
    'Lynx lynx mitochondrion', 'Macaca mulatta vestita mitochondrion', 'Marmota monax isolate',
    'Martes martes isolate', 'Mephitis mephitis voucher', 'Meriones unguiculatus mitochondrion',
    'Microtus arvalis isolate', 'Mus musculus strain', 'Mustela erminea isolate',
    'Mustela putorius furo isolate', 'Myocastor coypus mitochondrion', 'Myotis lucifugus mitochondrion',
    'Nasalis larvatus mitochondrion', 'Neotoma albigula isolate', 'Nyctereutes procyonoides mitochondrion',
    'Octodon degus voucher', 'Otaria byronia mitochondrion', 'Ovis aries breed',
    'Pan troglodytes mitochondrion', 'Panthera leo isolate', 'Panthera tigris amoyensis mitochondrion',
    'Papio hamadryas hamadryas isolate awash', 'Peromyscus maniculatus voucher',
    'Phodopus sungorus voucher', 'Pipistrellus pipistrellus mitochondrion', 'Pongo pygmaeus mitochondrion',
    'Procyon lotor mitochondrial', 'Pteropus vampyrus mitochondrion',
    'Rangifer tarandus platyrhyncus genome assembly', 'Rattus norvegicus strain',
    'Rattus rattus mitochondrion', 'Rhinolophus ferrumequinum quelpartis mitochondrion',
    'Saguinus oedipus mitochondrion', 'Saimiri sciureus mitochondrion',
    'Trachypithecus francoisi mitochondrion', 'Ursus maritimus complete mitochondrial genome',
    'Vespertilio murinus mitochondrion', 'Vulpes vulpes mitochondrion',
])
cytb_species = set([
    'Acomys cahirinus isolate', 'Ailurus fulgens mitochondrion', 'Alouatta palliata isolate',
    'Antilocapra americana isolate', 'Aotus trivirgatus isolate', 'Ateles geoffroyi isolate',
    'Bassariscus astutus voucher', 'Bison bison mitochondrion', 'Bos taurus breed',
    'Brachyteles arachnoides mitochondrion sequence', 'Bubalus bubalis isolate',
    'Callithrix jacchus mitochondrion', 'Camelus dromedarius isolate', 'Capra hircus haplogroup',
    'Carcharodon carcharias voucher', 'Carlito syrichta mitochondrion', 'Castor canadensis mitochondrion',
    'Chinchilla lanigera mitochondrion', 'Chlorocebus aethiops mitochondrion',
    'Colobus guereza mitochondrion', 'Cricetulus griseus mitochondrion', 'Dama dama isolate',
    'Dipodomys ordii mitochondrion sequence', 'Enhydra lutris mitochondrion',
    'Eptesicus fuscus mitochondrion', 'Equus asinus isolate', 'Equus caballus voucher',
    'Equus przewalskii mitochondrion', 'Felis catus isolate', 'Gazella gazella isolate',
    'Giraffa camelopardalis camelopardalis isolate', 'Gorilla beringei isolate',
    'Gorilla gorilla gorilla mitochondrion', 'Gulo gulo mitochondrion',
    'Herpestes javanicus mitochondrion', 'Hippopotamus amphibius mitochondrion', 'Hyaena hyaena isolate',
    'Ictidomys tridecemlineatus mitochondrion', 'Lagothrix lagotricha mitochondrion',
    'Lemur catta isolate m', 'Lutra lutra isolate', 'Lynx lynx mitochondrion',
    'Macaca mulatta vestita mitochondrion', 'Martes martes isolate', 'Mephitis mephitis voucher',
    'Meriones unguiculatus mitochondrion', 'Microtus arvalis isolate', 'Mus musculus strain',
    'Mustela erminea isolate', 'Mustela putorius furo isolate', 'Myocastor coypus mitochondrion',
    'Myotis lucifugus mitochondrion', 'Nasalis larvatus mitochondrion', 'Neotoma albigula isolate',
    'Nyctereutes procyonoides mitochondrion', 'Octodon degus voucher', 'Otaria byronia mitochondrion',
    'Ovis aries mitochondrion', 'Pan troglodytes mitochondrion', 'Panthera leo isolate',
    'Panthera tigris amoyensis mitochondrion', 'Papio hamadryas hamadryas isolate awash',
    'Peromyscus maniculatus voucher', 'Phodopus sungorus voucher', 'Pipistrellus pipistrellus mitochondrion',
    'Pongo pygmaeus mitochondrion', 'Procyon lotor mitochondrion', 'Pteropus vampyrus mitochondrion',
    'Rangifer tarandus platyrhyncus genome assembly', 'Rattus norvegicus strain',
    'Rattus rattus mitochondrion', 'Rhinolophus ferrumequinum quelpartis mitochondrion',
    'Saguinus oedipus mitochondrion', 'Saimiri sciureus mitochondrion',
    'Trachypithecus francoisi mitochondrion', 'Ursus maritimus mitochondrion',
    'Vespertilio murinus mitochondrion',
])

In [ ]:
def normalize_species_name(raw_name):
    """Strip gene/source qualifiers and return 'genus species' in lowercase."""
    name = raw_name.lower().strip()
    # Remove common qualifiers
    name = re.sub(
        r'\b(recombination activating|mitochondrion.*?|isolate.*?|voucher|sequence'
        r'|gene for .+?|strain|breed|haplogroup|type.*?|complete mitochondrial genome'
        r'|genome assembly|apolipoprotein|breast cancer|mitochondrial.*?'
        r'|cytb.*?|coi.*?|mrna|rna|cdna|brca.*?)\b',
        '', name,
    )
    name = re.sub(r'[^a-z\s\-]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    parts = name.split()
    if len(parts) >= 2:
        return f"{parts[0]} {parts[1]}"
    return parts[0] if parts else ''


def normalize_species(species_set):
    return {normalize_species_name(s) for s in species_set}


# Normalize all five sets and find intersection
normalized = {
    'rag1':  normalize_species(rag1_species),
    'brca1': normalize_species(brca1_species),
    'apob':  normalize_species(apob_species),
    'coi':   normalize_species(coi_species),
    'cytb':  normalize_species(cytb_species),
}
common_species = (
    normalized['rag1'] & normalized['brca1'] & normalized['apob']
    & normalized['coi'] & normalized['cytb']
)
print(f"Common species across all five genes: {len(common_species)}")
for sp in sorted(common_species):
    print(f"  - {sp}")

# ── Extract common-species sequences into per-gene FASTAs ────────────────────
def get_species_name(record, species_set):
    desc = record.description.lower()
    for sp in species_set:
        if sp in desc:
            return sp
    return None


os.makedirs(EXTRACTED_DIR, exist_ok=True)
gene_file_map = {
    'rag1':  os.path.join(FILTERED_DIR, "RAG1.filtered.fasta"),
    'brca1': os.path.join(FILTERED_DIR, "BRCA1.filtered.fasta"),
    'apob':  os.path.join(FILTERED_DIR, "APOB.filtered.fasta"),
    'coi':   os.path.join(FILTERED_DIR, "COI.filtered.fasta"),
    'cytb':  os.path.join(FILTERED_DIR, "CytB.filtered.fasta"),
}

for gene, file_path in gene_file_map.items():
    extracted = [
        r for r in SeqIO.parse(file_path, "fasta")
        if get_species_name(r, common_species)
    ]
    out_path = os.path.join(EXTRACTED_DIR, f"{gene}_common_species.fasta")
    SeqIO.write(extracted, out_path, "fasta")
    print(f"[✓] {gene}: {len(extracted)} sequences → {out_path}")

## 7. Sequence Alignment (MAFFT + TrimAl)
Requires MAFFT and TrimAl installed. Update `MAFFT_PATH` / `TRIMAL_PATH` in the config cell if they are not on your PATH.

In [ ]:
def ensure_unique_ids(fasta_path, tmp_path):
    """Rename sequence IDs to seq1, seq2, … for PHYLIP compatibility."""
    records = [
        SeqRecord(r.seq, id=f"seq{i+1}", name=f"seq{i+1}", description="")
        for i, r in enumerate(SeqIO.parse(fasta_path, "fasta"))
    ]
    SeqIO.write(records, tmp_path, "fasta")


def trim_alignment(input_fasta, output_fasta_trimmed, trimal_path=TRIMAL_PATH):
    """Run TrimAl on an aligned FASTA file."""
    result = subprocess.run(
        [trimal_path, "-in", input_fasta, "-out", output_fasta_trimmed, "-automated1"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"[!] TrimAl failed:\n{result.stderr}")
    else:
        print(f"[✓] Trimmed → {output_fasta_trimmed}")


def clean_and_export_phylip(trimmed_fasta, output_phylip):
    """Write a trimmed FASTA to PHYLIP-relaxed format with clean IDs."""
    records = [
        SeqRecord(r.seq, id=f"seq{i+1}", name=f"seq{i+1}", description="")
        for i, r in enumerate(SeqIO.parse(trimmed_fasta, "fasta"))
    ]
    alignment = MultipleSeqAlignment(records)
    with open(output_phylip, "w") as fh:
        AlignIO.write(alignment, fh, "phylip-relaxed")
    print(f"[✓] PHYLIP → {output_phylip}")


def align_with_mafft(input_fasta, aligned_fasta, trimal_path=TRIMAL_PATH, mafft_path=MAFFT_PATH):
    """Align with MAFFT, trim with TrimAl, and export PHYLIP."""
    result = subprocess.run(
        [mafft_path, "--auto", input_fasta],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"[!] MAFFT failed for {input_fasta}:\n{result.stderr}")
        return

    with open(aligned_fasta, "w") as fh:
        fh.write(result.stdout)
    print(f"[✓] Aligned FASTA → {aligned_fasta}")

    trimmed_fasta = aligned_fasta.replace(".fasta", ".trimmed.fasta")
    trim_alignment(aligned_fasta, trimmed_fasta, trimal_path)

    phylip_path = aligned_fasta.replace(".fasta", "_clean.phy")
    clean_and_export_phylip(trimmed_fasta, phylip_path)

In [ ]:
os.makedirs(ALIGNED_DIR, exist_ok=True)

for gene in GENES:
    input_file  = os.path.join(EXTRACTED_DIR, f"{gene}_common_species.fasta")
    aligned_out = os.path.join(ALIGNED_DIR,   f"{gene}_aligned.fasta")
    print(f"\n[→] Aligning {gene}")
    align_with_mafft(input_file, aligned_out)

## 8. Feature Engineering

### 8a. K-mer Frequencies

In [ ]:
def kmer_frequencies(seq, k=KMER_K):
    """Return a normalised k-mer frequency vector for a DNA sequence."""
    seq = seq.upper().replace("-", "").replace("N", "")
    valid = {"A", "C", "G", "T"}
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1) if set(seq[i:i+k]).issubset(valid)]
    counts = Counter(kmers)
    all_kmers = ["".join(p) for p in product("ACGT", repeat=k)]
    vec = np.array([counts.get(km, 0) for km in all_kmers], dtype=float)
    total = vec.sum()
    return vec / total if total > 0 else np.zeros(len(all_kmers))


os.makedirs(KMER_DIR, exist_ok=True)

for gene in GENES:
    fasta_path = os.path.join(ALIGNED_DIR, f"{gene}_aligned.trimmed.fasta")
    rows, species_names = [], []

    for record in SeqIO.parse(fasta_path, "fasta"):
        species = ACCESSION_TO_SPECIES.get(record.id, record.id)
        species_names.append(species)
        rows.append(kmer_frequencies(str(record.seq)))

    df = pd.DataFrame(rows)
    df.insert(0, "Species", species_names)
    out_csv = os.path.join(KMER_DIR, f"{gene}_kmer_frequencies.csv")
    df.to_csv(out_csv, index=False)
    print(f"[✓] {gene.upper()}: {df.shape[0]} sequences, {df.shape[1]-1} k-mers → {out_csv}")

# Validation
print("\n--- NaN check ---")
for gene in GENES:
    df = pd.read_csv(os.path.join(KMER_DIR, f"{gene}_kmer_frequencies.csv"))
    print(f"  {gene.upper()} → NaNs: {df.isnull().values.any()}")

### 8b. One-Hot Encoding

In [ ]:
ONE_HOT_MAP = {
    "A": [1,0,0,0], "C": [0,1,0,0], "G": [0,0,1,0], "T": [0,0,0,1], "-": [0,0,0,0],
}

def one_hot_encode(seq):
    """Encode a DNA/alignment sequence as a (L, 4) one-hot array."""
    return np.array([ONE_HOT_MAP.get(b.upper(), [0,0,0,0]) for b in seq])


os.makedirs(ONEHOT_DIR, exist_ok=True)

for gene in GENES:
    fasta_path = os.path.join(ALIGNED_DIR, f"{gene}_aligned.trimmed.fasta")
    arrays = [one_hot_encode(str(r.seq)) for r in SeqIO.parse(fasta_path, "fasta")]
    arr = np.array(arrays)
    out_path = os.path.join(ONEHOT_DIR, f"{gene}_onehot.npy")
    np.save(out_path, arr)
    print(f"[✓] {gene.upper()}: shape={arr.shape} → {out_path}")

# Validation
print("\n--- Unique value check (should be 0 and 1 only) ---")
for gene in GENES:
    data = np.load(os.path.join(ONEHOT_DIR, f"{gene}_onehot.npy"))
    print(f"  {gene.upper()} → unique values: {np.unique(data)}")

### 8c. BioVec Embeddings (Word2Vec on k-mers)

In [ ]:
def generate_kmers(seq, k=KMER_K):
    return [str(seq[i:i+k]) for i in range(len(seq) - k + 1)]


def get_sequence_embedding(kmers, model):
    vecs = [model.wv[k] for k in kmers if k in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)


# Step 1: Build k-mer corpus from all genes
corpus = []
for gene in GENES:
    fasta_path = os.path.join(ALIGNED_DIR, f"{gene}_aligned.trimmed.fasta")
    for record in SeqIO.parse(fasta_path, "fasta"):
        corpus.append(generate_kmers(str(record.seq)))

# Step 2: Train BioVec model
biovec_model = Word2Vec(
    sentences=corpus, vector_size=BIOVEC_DIM, window=5, min_count=1, sg=1, workers=4
)
print(f"[✓] BioVec model trained on {len(corpus)} sequences.")

# Step 3: Compute and save embeddings per gene
os.makedirs(BIOVEC_DIR, exist_ok=True)

for gene in GENES:
    fasta_path = os.path.join(ALIGNED_DIR, f"{gene}_aligned.trimmed.fasta")
    embeddings = [
        get_sequence_embedding(generate_kmers(str(r.seq)), biovec_model)
        for r in SeqIO.parse(fasta_path, "fasta")
    ]
    out_path = os.path.join(BIOVEC_DIR, f"{gene}_biovec.npy")
    np.save(out_path, np.array(embeddings))
    print(f"[✓] {gene.upper()}: {len(embeddings)} embeddings → {out_path}")

# Validation
print("\n--- Sequence vs embedding count check ---")
for gene in GENES:
    n_seqs = sum(1 for _ in SeqIO.parse(os.path.join(ALIGNED_DIR, f"{gene}_aligned.trimmed.fasta"), "fasta"))
    emb = np.load(os.path.join(BIOVEC_DIR, f"{gene}_biovec.npy"))
    match = "✓" if n_seqs == emb.shape[0] else "✗"
    print(f"  [{match}] {gene.upper()}: {n_seqs} seqs, {emb.shape[0]} embeddings")

## 9. Phylogenetic Tree & Distance Matrix

In [ ]:
common_species_list = sorted(common_species)

# ── Download tree from Open Tree of Life ─────────────────────────────────────
match_url = "https://api.opentreeoflife.org/v3/tnrs/match_names"
match_resp = requests.post(match_url, json={"names": common_species_list})
match_resp.raise_for_status()
match_data = match_resp.json()

ott_ids = []
for result in match_data.get("results", []):
    matches = result.get("matches", [])
    if matches:
        ott_ids.append(matches[0]["taxon"]["ott_id"])

induced_url = "https://api.opentreeoflife.org/v3/tree_of_life/induced_subtree"
tree_resp = requests.post(induced_url, json={"ott_ids": ott_ids})
tree_resp.raise_for_status()
tree_data = tree_resp.json()

newick_str = tree_data.get("newick", "")
if newick_str:
    with open("phylogenetic_tree.nwk", "w") as f:
        f.write(newick_str)
    print("Saved → phylogenetic_tree.nwk")
else:
    print("No Newick data returned.")

In [ ]:
# ── Pairwise distances ────────────────────────────────────────────────────────
tree = Tree("phylogenetic_tree.nwk", format=1, quoted_node_names=True)
leaves = [leaf.name for leaf in tree]

distance_rows = [
    (leaves[i], leaves[j], tree.get_distance(leaves[i], leaves[j]))
    for i in range(len(leaves))
    for j in range(i + 1, len(leaves))
] + [(sp, sp, 0.0) for sp in leaves]

distance_df = pd.DataFrame(distance_rows, columns=["species_1", "species_2", "distance"])
distance_df.to_csv("pairwise_distances.csv", index=False)
print(f"Pairwise distances: {distance_df.shape} → pairwise_distances.csv")

# ── Square distance matrix ────────────────────────────────────────────────────
square_dist_df = distance_df.pivot(index="species_1", columns="species_2", values="distance")
square_dist_df = square_dist_df.combine_first(square_dist_df.T)   # fill symmetric part
square_dist_df.to_csv("square_distance_matrix.csv")

nan_count = square_dist_df.isna().sum().sum()
print(f"Square matrix: {square_dist_df.shape} | NaNs: {nan_count} → square_distance_matrix.csv")

## 10. Metadata Preparation

In [ ]:
SPECIES_LIST = [
    "Callithrix jacchus", "Capra hircus", "Castor canadensis", "Chinchilla lanigera",
    "Dipodomys ordii", "Eptesicus fuscus", "Equus asinus", "Equus przewalskii",
    "Gorilla gorilla", "Hippopotamus amphibius", "Hyaena hyaena", "Ictidomys tridecemlineatus",
    "Lemur catta", "Macaca mulatta", "Mus musculus", "Myocastor coypus", "Nyctereutes procyonoides",
    "Ovis aries", "Panthera leo", "Panthera tigris", "Rattus rattus", "Rhinolophus ferrumequinum",
    "Trachypithecus francoisi", "Ursus maritimus",
]

# ── Load and filter metadata CSV ──────────────────────────────────────────────
metadata_df = pd.read_csv("mammal_metadata.csv")
filtered_meta = metadata_df[metadata_df["Scientific_Name"].isin(SPECIES_LIST)].copy()
filtered_meta.to_csv("filtered_mammal_metadata.csv", index=False)
print(f"Filtered metadata: {filtered_meta.shape} → filtered_mammal_metadata.csv")

# ── Geocoding ─────────────────────────────────────────────────────────────────
def simplify_location(distribution):
    if pd.isna(distribution) or "Domesticated" in str(distribution):
        return "World"
    return distribution.split("|")[0]


def get_lat_long(location_name, geocode_fn):
    try:
        loc = geocode_fn(location_name)
        return pd.Series([loc.latitude, loc.longitude]) if loc else pd.Series([None, None])
    except Exception:
        return pd.Series([None, None])


geolocator = Nominatim(user_agent="species_geocoder")
geocode    = RateLimiter(geolocator.geocode, min_delay_seconds=1)

filtered_meta["location_for_geocoding"] = filtered_meta["countryDistribution"].apply(simplify_location)
filtered_meta[["latitude", "longitude"]] = filtered_meta["location_for_geocoding"].apply(
    lambda x: get_lat_long(x, geocode)
)

# ── IUCN ordinal encoding ─────────────────────────────────────────────────────
filtered_meta["iucn_ordinal"] = filtered_meta["iucnStatus"].map(IUCN_ORDER)

filtered_meta.to_csv("updated_species_with_latlong.csv", index=False)
print(f"Metadata with geocoding + IUCN ordinal saved → updated_species_with_latlong.csv")
print(filtered_meta.isnull().sum())

## 11. Hierarchical Clustering
Cluster species by their concatenated BioVec embeddings and evaluate against biogeographic region and IUCN status.

In [ ]:
# ── Load embeddings and metadata ──────────────────────────────────────────────
embeddings = [np.load(os.path.join(BIOVEC_DIR, f"{gene}_biovec.npy")) for gene in GENES]
X = np.concatenate(embeddings, axis=1)

metadata = pd.read_csv("updated_species_with_latlong.csv")
species_names = metadata["Scientific_Name"].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Hierarchical clustering ───────────────────────────────────────────────────
dist_matrix    = pdist(X_scaled, metric="cosine")
linkage_matrix = linkage(dist_matrix, method="average")

threshold = 0.7 * max(linkage_matrix[:, 2])
metadata["cluster_hier"] = fcluster(linkage_matrix, t=threshold, criterion="distance")


def plot_colored_dendrogram(color_by, title, palette_name="bright"):
    unique_vals  = metadata[color_by].unique()
    color_map    = dict(zip(unique_vals, sns.color_palette(palette_name, len(unique_vals))))
    label_colors = metadata[color_by].map(color_map).tolist()

    plt.figure(figsize=(18, 8))
    dendro = dendrogram(linkage_matrix, labels=species_names, leaf_rotation=90, leaf_font_size=10)
    ax = plt.gca()
    for lbl, color in zip(ax.get_xmajorticklabels(), label_colors):
        lbl.set_color(color)
    plt.title(title)
    plt.tight_layout()
    plt.show()


# ── Plots ─────────────────────────────────────────────────────────────────────
plot_colored_dendrogram("biogeographicRealm", "Dendrogram Colored by Biogeographic Region")
plot_colored_dendrogram("iucnStatus",         "Dendrogram Colored by IUCN Conservation Status")

# ── Contingency tables ────────────────────────────────────────────────────────
print("\nContingency Table: Cluster vs Region")
print(pd.crosstab(metadata["cluster_hier"], metadata["biogeographicRealm"]))
print("\nContingency Table: Cluster vs Conservation Status")
print(pd.crosstab(metadata["cluster_hier"], metadata["iucnStatus"]))

# ── Silhouette score ──────────────────────────────────────────────────────────
score = silhouette_score(X_scaled, metadata["cluster_hier"], metric="cosine")
print(f"\nSilhouette Score (Cosine): {score:.4f}")

## 12. Multi-Input CNN Classification

In [ ]:
class GeneCNN(nn.Module):
    """1D CNN branch for a single gene's one-hot encoded sequence."""
    def __init__(self, input_channels=4, output_dim=64):
        super().__init__()
        self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.pool  = nn.AdaptiveMaxPool1d(1)
        self.fc    = nn.Linear(64, output_dim)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return self.fc(self.pool(x).squeeze(-1))


class MetadataEncoder(nn.Module):
    """MLP encoder for numeric metadata features (lat, lon, IUCN ordinal)."""
    def __init__(self, input_dim=3, output_dim=64):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, output_dim),
        )

    def forward(self, x):
        return self.fc(x)


class MultiGeneHybridModel(nn.Module):
    """Five parallel GeneCNN branches fused with a MetadataEncoder for classification."""
    def __init__(self, gene_output_dim=64, meta_output_dim=64, num_classes=NUM_CLASSES):
        super().__init__()
        self.gene_cnns        = nn.ModuleList([GeneCNN(output_dim=gene_output_dim) for _ in range(5)])
        self.metadata_encoder = MetadataEncoder(output_dim=meta_output_dim)
        self.classifier       = nn.Sequential(
            nn.Linear(5 * gene_output_dim + meta_output_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, gene_inputs, meta_input):
        gene_embeds = [cnn(x) for cnn, x in zip(self.gene_cnns, gene_inputs)]
        combined    = torch.cat(gene_embeds + [self.metadata_encoder(meta_input)], dim=1)
        return self.classifier(combined)

In [ ]:
class MultiGeneDataset(Dataset):
    """Dataset wrapping per-gene one-hot arrays, metadata, and cluster labels."""
    def __init__(self, gene_arrays, metadata_df, labels):
        # gene_arrays: list of (N, L, 4) numpy arrays; transpose to (N, 4, L) for Conv1d
        self.gene_tensors = [
            torch.tensor(g.transpose(0, 2, 1), dtype=torch.float32) for g in gene_arrays
        ]
        scaler = StandardScaler()
        meta   = scaler.fit_transform(metadata_df[["latitude", "longitude", "iucn_ordinal"]])
        self.meta   = torch.tensor(meta,   dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.gene_tensors[0].shape[0]

    def __getitem__(self, idx):
        return [g[idx] for g in self.gene_tensors], self.meta[idx], self.labels[idx]


def train_model(model, dataloader, epochs=EPOCHS, lr=LEARNING_RATE):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for gene_inputs, meta_input, labels in dataloader:
            optimizer.zero_grad()
            loss = criterion(model(gene_inputs, meta_input), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f}")


def evaluate_model(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for gene_inputs, meta_input, labels in dataloader:
            _, predicted = torch.max(model(gene_inputs, meta_input), dim=1)
            all_preds.append(predicted.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
gene_arrays = [np.load(os.path.join(ONEHOT_DIR, f"{gene}_onehot.npy")) for gene in GENES]
assert all(g.shape[0] == gene_arrays[0].shape[0] for g in gene_arrays), "Sample count mismatch!"

df_meta   = pd.read_csv("updated_species_with_latlong.csv")
dist_df   = pd.read_csv("square_distance_matrix.csv", index_col=0)
labels    = KMeans(n_clusters=NUM_CLASSES, random_state=42).fit_predict(dist_df.values)

dataset    = MultiGeneDataset(gene_arrays, df_meta, labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ── Train ─────────────────────────────────────────────────────────────────────
model = MultiGeneHybridModel()
train_model(model, dataloader)

# ── Evaluate ──────────────────────────────────────────────────────────────────
evaluate_model(model, dataloader)

In [ ]:
# ── Cluster quality vs phylogenetic family ────────────────────────────────────
ari = adjusted_rand_score(metadata["family"], labels)
nmi = normalized_mutual_info_score(metadata["family"], labels)
print(f"ARI: {ari:.4f}  |  NMI: {nmi:.4f}")

# ── Visualise phylogenetic tree ────────────────────────────────────────────────
bio_tree = Phylo.read("phylogenetic_tree.nwk", "newick")
for clade in bio_tree.get_terminals():
    if clade.name and len(clade.name) > 30:
        clade.name = clade.name[:30] + "..."
for clade in bio_tree.get_nonterminals():
    if clade.name:
        clade.name = re.sub(r"ott\d+", "", clade.name)
Phylo.draw_ascii(bio_tree)